# Phase 2: Data Preprocessing Pipeline (Refactored)
Final Blueprint: Sub-Minute Aggregation with Normalized RUL

This pipeline resolves the "Mean Prediction Collapse" (flat ~0.5, negative R2)
by implementing five architectural corrections derived from a comprehensive
trade-off analysis of three independent audits (Claude, DeepSeek, Qwen).

Key Changes from Previous Pipeline:
    1. REMOVED  - Negative Transfer Alignment (Pearson correlation flip)
    2. REFACTORED - Ground truth: smoothed CUSUM + Normalized RUL (no plateau)
    3. ADDED   - Sub-minute aggregation (30s hop) for macroscopic temporal context
    4. FIXED   - Per-bearing MinMax scaling applied AFTER aggregation
    5. NOTED   - Over-regularization guidelines for training phase

Pipeline Flow:
    Raw Segments -> 14 Features per segment
    -> Sub-Minute Aggregation (mean+std, 30s hop) -> 28 Features per timestep
    -> Smoothed CUSUM (metadata) + Normalized RUL (target)
    -> Per-Bearing MinMax Scaling
    -> Sliding Sequence Windows [10, 20, 30]
    -> CSV output for LSTM training

In [ ]:
import os
import sys
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from typing import List, Dict, Tuple
from tqdm import tqdm
from numpy.lib.stride_tricks import sliding_window_view

In [ ]:
sys.path.append("../src")
from CrossDomainFeatureExtractor import CrossDomainFeatureExtractor
from ConstructGroundTruthUsingCUSUM import UnivariateCUSUMDetector

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
INPUT_PATH = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets\Processed_Data\Downsampled"
OUTPUT_PATH = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets\Processed_Data\LSTM_Inputs"
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
# Sequence window sizes: with sub-minute rows, these cover 5-15 min physical
WINDOW_SIZES = [10, 20, 30]

In [ ]:
# Signal processing parameters
XJTU_SAMPLING_RATE = 25600.0
MAX_FREQ_HZ = 1280.0
SEGMENT_LENGTH = 1024
STRIDE = 512

In [ ]:
# CUSUM parameters (conservative to prevent premature triggering)
CUSUM_BASELINE_RATIO = 0.2
CUSUM_K_FACTOR = 0.5
CUSUM_H_FACTOR = 8.0
CUSUM_SMOOTHING_WINDOW = 10

In [ ]:
def _compute_mean_std(
    values: np.ndarray, col_names: List[str]
) -> Dict[str, float]:
    """
    Compute mean and std for each feature column from a 2D numpy array.

    Args:
        values: 2D array of shape (n_segments, n_features).
        col_names: List of original feature column names.

    Returns:
        Dictionary with keys '{col}_mean' and '{col}_std' for each feature.
    """
    row = {}
    for j, col in enumerate(col_names):
        col_vals = values[:, j]
        row[f"{col}_mean"] = float(np.mean(col_vals))
        if len(col_vals) > 1:
            row[f"{col}_std"] = float(np.std(col_vals, ddof=1))
        else:
            row[f"{col}_std"] = 0.0
    return row

In [ ]:
print(f"Input Path:  {os.path.abspath(INPUT_PATH)}")
print(f"Output Path: {os.path.abspath(OUTPUT_PATH)}")
print(f"Window Sizes: {WINDOW_SIZES}")
print(f"Segment: {SEGMENT_LENGTH} samples, Stride: {STRIDE} samples")

In [ ]:
# ============================================================================
# PHASE 1: RAW FEATURE EXTRACTION (SEGMENT-LEVEL)
# ============================================================================
def segment_and_extract(file_path: str, bearing_id: str) -> pd.DataFrame:
    """
    Extract 14 physics-based features from overlapping raw signal segments.

    Returns UNSCALED, UNFLIPPED segment-level features. No CUSUM, no HI,
    no RUL, no scaling. These operations are deferred to downstream
    functions that operate on aggregated sub-minute data.

    Args:
        file_path: Path to the aggregated bearing pickle file.
        bearing_id: Canonical bearing identifier (e.g., "Bearing1_1").

    Returns:
        DataFrame with columns: [14 raw features, Minute, Bearing_ID].
        Empty DataFrame if extraction fails entirely.
    """
    df_raw = pd.read_pickle(file_path)
    extractor = CrossDomainFeatureExtractor(
        sampling_rate=XJTU_SAMPLING_RATE,
        max_freq_hz=MAX_FREQ_HZ
    )
    features_list = []

    for _, row in tqdm(
        df_raw.iterrows(), total=len(df_raw),
        desc=f"Extracting {bearing_id}", leave=False
    ):
        try:
            h_sig = np.array(row["H_Signal"], dtype=float)
            if len(h_sig) < SEGMENT_LENGTH:
                continue
            segments = sliding_window_view(
                h_sig, window_shape=SEGMENT_LENGTH
            )[::STRIDE]
            for seg in segments:
                feats = extractor.extract_all_features(seg)
                feats["Minute"] = row["Minute"]
                feats["Bearing_ID"] = bearing_id
                features_list.append(feats)
        except Exception:
            continue

    df_features = pd.DataFrame(features_list)
    return df_features

In [ ]:
# ============================================================================
# PHASE 2: SUB-MINUTE AGGREGATION (30-SECOND HOP)
# ============================================================================
def aggregate_with_subminute_hop(df_segments: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate segment-level features into sub-minute temporal resolution.

    Uses overlapping 1-minute windows with a 30-second hop to double the
    effective dataset size while providing macroscopic temporal context.

    Strategy:
        - At integer minute m: aggregate all segments from minute m.
        - At half-minute m+0.5: aggregate segments from minutes m AND m+1.
        - Result: (2*N - 1) rows from N original minutes.

    For each of the 14 raw features, computes:
        - Mean (central tendency of the vibration characteristic)
        - Std  (within-window variability / micro-dynamics)

    Output: 28 features per timestep.

    Args:
        df_segments: Segment-level features from segment_and_extract().

    Returns:
        DataFrame with columns: [28 aggregated features, Time_Index, Bearing_ID].
    """
    feat_cols = [
        c for c in df_segments.columns if c not in ["Minute", "Bearing_ID"]
    ]
    minutes = sorted(df_segments["Minute"].unique())
    bearing_id = df_segments["Bearing_ID"].iloc[0]

    # Pre-group segments by minute for efficient lookup
    grouped = {
        m: df_segments.loc[df_segments["Minute"] == m, feat_cols].values
        for m in minutes
    }

    agg_rows = []

    for i, m in enumerate(minutes):
        seg_vals = grouped[m]

        # --- Integer minute: single-minute aggregation ---
        row = _compute_mean_std(seg_vals, feat_cols)
        row["Time_Index"] = float(m)
        row["Bearing_ID"] = bearing_id
        agg_rows.append(row)

        # --- Half-minute: overlapping two-minute aggregation ---
        if i < len(minutes) - 1:
            m_next = minutes[i + 1]
            combined = np.vstack([seg_vals, grouped[m_next]])

            row_half = _compute_mean_std(combined, feat_cols)
            row_half["Time_Index"] = float(m) + 0.5
            row_half["Bearing_ID"] = bearing_id
            agg_rows.append(row_half)

    df_agg = pd.DataFrame(agg_rows)
    df_agg = df_agg.sort_values("Time_Index").reset_index(drop=True)
    df_agg = df_agg.fillna(0.0)

    return df_agg

In [ ]:
# ============================================================================
# PHASE 3: GROUND TRUTH CONSTRUCTION
# ============================================================================
def construct_ground_truth(df_agg: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
    """
    Construct ground truth labels: Normalized RUL + CUSUM metadata.

    Two-stage process:
        1. CUSUM Detection:
           Smooths td_rms_mean with a rolling window, then applies
           conservative CUSUM.

        2. Piecewise-Linear RUL Target with Safety Guard:
           HI(t) = 1.0 for t <= t_cp (healthy phase)
           HI(t) = 1.0 - (t - t_cp) / (t_f - t_cp) for t > t_cp (degradation)
           Guard: t_cp = max(change_point, 0.3 * t_f) to prevent early noise trigger.

    Args:
        df_agg: Sub-minute aggregated features from aggregate_with_subminute_hop().

    Returns:
        Tuple of:
            - DataFrame with added Health_Index, Target_RUL, Change_Point columns.
            - Metadata dictionary for logging.
    """
    df_out = df_agg.copy()
    n_total = len(df_out)
    t_f = n_total - 1
    bearing_id = df_out["Bearing_ID"].iloc[0]

    # --- Stage 1: CUSUM on smoothed RMS ---
    rms_col = "td_rms_mean"
    if rms_col not in df_out.columns:
        available = [c for c in df_out.columns if "rms" in c.lower()]
        raise ValueError(
            f"Column '{rms_col}' not found. RMS-related columns: {available}"
        )

    smoothed_rms = df_out[rms_col].rolling(
        window=CUSUM_SMOOTHING_WINDOW, min_periods=1, center=True
    ).mean()

    detector = UnivariateCUSUMDetector(
        baseline_ratio=CUSUM_BASELINE_RATIO,
        k_factor=CUSUM_K_FACTOR,
        h_factor=CUSUM_H_FACTOR
    )
    change_point, _ = detector.fit_predict(smoothed_rms.values)

    # --- Stage 2: Piecewise-Linear HI Target with Safety Guard ---
    t_cp = max(int(change_point), int(0.3 * t_f))
    indices = np.arange(n_total, dtype=np.float64)
    
    health_indices = np.ones(n_total)
    degradation_len = t_f - t_cp
    if degradation_len > 0:
        degradation_mask = indices > t_cp
        health_indices[degradation_mask] = 1.0 - (indices[degradation_mask] - t_cp) / degradation_len
    elif t_f > 0:
        health_indices = 1.0 - (indices / t_f)
    else:
        health_indices = np.zeros(n_total)

    df_out["Health_Index"] = np.clip(health_indices, 0.0, 1.0)
    df_out["Target_RUL"] = np.maximum(0.0, t_f - indices)
    df_out["Change_Point"] = int(t_cp)

    # Enforce hard boundary constraints
    df_out.iloc[-1, df_out.columns.get_loc("Health_Index")] = 0.0
    df_out.iloc[-1, df_out.columns.get_loc("Target_RUL")] = 0.0

    metadata = {
        "Bearing_ID": bearing_id,
        "Total_Timesteps": n_total,
        "Total_Minutes": int(df_out["Time_Index"].max()),
        "Change_Point_Index": int(t_cp),
        "Change_Point_Minute": float(
            df_out["Time_Index"].iloc[t_cp]
        ) if t_cp < n_total else -1.0,
    }

    return df_out, metadata

In [ ]:
# ============================================================================
# PHASE 4: SLIDING SEQUENCE WINDOWS FOR LSTM
# ============================================================================
def create_sliding_windows(
    data_df: pd.DataFrame, window_size: int
) -> pd.DataFrame:
    """
    Create chronological sliding windows for LSTM input.

    Each window captures 'window_size' consecutive sub-minute timesteps.
    With 30-second hop aggregation:
        - window_size=10 covers  5 minutes of physical time
        - window_size=20 covers 10 minutes of physical time
        - window_size=30 covers 15 minutes of physical time

    Target alignment: Health Index and RUL of the LAST timestep in each
    window are used as the prediction target (causal alignment).

    Args:
        data_df: Scaled, aggregated features with ground truth labels.
        window_size: Number of consecutive timesteps per LSTM sequence.

    Returns:
        Flattened DataFrame: each row is one LSTM sample.
        Feature columns: {feature_name}_t{timestep_index}.
    """
    if len(data_df) <= window_size:
        return pd.DataFrame()

    meta_cols = [
        "Change_Point", "Health_Index", "Target_RUL",
        "Time_Index", "Bearing_ID"
    ]
    feature_cols = sorted(
        [c for c in data_df.columns if c not in meta_cols]
    )

    X_data = data_df[feature_cols].values
    X_windows = sliding_window_view(
        X_data, window_shape=window_size, axis=0
    )  # Shape: (N - W + 1, n_features, W)

    n_windows = X_windows.shape[0]
    targets_hi = data_df["Health_Index"].values[window_size - 1:]
    targets_rul = data_df["Target_RUL"].values[window_size - 1:]
    time_indices = data_df["Time_Index"].values[window_size - 1:]
    bearing_id = data_df["Bearing_ID"].values[-1]

    out_rows = []
    for i in range(n_windows):
        row_record = {}
        for f_idx, feat in enumerate(feature_cols):
            for t in range(window_size):
                row_record[f"{feat}_t{t}"] = X_windows[i, f_idx, t]
        row_record["Health_Index"] = targets_hi[i]
        row_record["Target_RUL"] = targets_rul[i]
        row_record["Original_Minute"] = time_indices[i]
        row_record["Bearing_ID"] = bearing_id
        out_rows.append(row_record)

    return pd.DataFrame(out_rows)

In [ ]:
# ============================================================================
# CROSS-VALIDATION FOLD DEFINITION
# ============================================================================
def get_stratified_folds() -> List[Dict[str, List[str]]]:
    """
    Define stratified cross-validation folds for XJTU-SY dataset.

    Each fold holds out one bearing per condition (3 bearings total),
    training on the remaining 12. This ensures each fold validates
    across all three operating conditions simultaneously.

    Returns:
        List of fold dictionaries with 'train' and 'val' bearing ID lists.
    """
    folds = [
        {"val": ["Bearing1_1", "Bearing2_1", "Bearing3_1"]},
        {"val": ["Bearing1_2", "Bearing2_2", "Bearing3_2"]},
        {"val": ["Bearing1_3", "Bearing2_3", "Bearing3_3"]},
        {"val": ["Bearing1_4", "Bearing2_4", "Bearing3_4"]},
        {"val": ["Bearing1_5", "Bearing2_5", "Bearing3_5"]},
    ]
    all_bearings = [
        f"Bearing{c}_{i}" for c in range(1, 4) for i in range(1, 6)
    ]
    for fold in folds:
        fold["train"] = [b for b in all_bearings if b not in fold["val"]]
    return folds

In [ ]:
# ============================================================================
# MAIN ORCHESTRATOR
# ============================================================================
def preprocess_and_slide():
    """
    Main preprocessing pipeline orchestrator.

    Execution order:
        1. Extract raw segment features for all bearings
        2. Aggregate to sub-minute resolution
        3. Construct ground truth (Piecewise-Linear + 30% Guard)
        4. Split folds and apply Global StandardScaler per fold
        5. Create sliding sequence windows per window size
        6. Split into train/val CSVs per fold

    Design Decisions:
        - Scaling is StandardScaler to preserve out-of-distribution severity (failure vs healthy).
        - Applied PER FOLD strictly fitting on train to avoid leakage.
    """
    aggregated_files = glob.glob(
        os.path.join(INPUT_PATH, "*", "*_aggregated.pkl")
    )
    if not aggregated_files:
        print(f"ERROR: No aggregated files found in {INPUT_PATH}")
        return

    print(f"Found {len(aggregated_files)} bearing files.")

    # ------------------------------------------------------------------
    # Step 1: Extract raw features for all bearings
    # ------------------------------------------------------------------
    all_segment_features = {}

    for file in tqdm(aggregated_files, desc="Phase 1: Feature Extraction"):
        raw_b_id = os.path.basename(file).replace("_aggregated.pkl", "")
        parent_dir = os.path.basename(os.path.dirname(file))
        c_idx = 1
        if "37.5" in parent_dir:
            c_idx = 2
        elif "40" in parent_dir:
            c_idx = 3
        b_num = raw_b_id.split("_")[-1] if raw_b_id.startswith("Bearing") else "1"
        bearing_id = f"Bearing{c_idx}_{b_num}"

        df_seg = segment_and_extract(file, bearing_id)
        if not df_seg.empty:
            all_segment_features[bearing_id] = df_seg
            print(f"  {bearing_id}: {len(df_seg)} segments, "
                  f"{df_seg['Minute'].nunique()} minutes")

    if not all_segment_features:
        print("ERROR: No features extracted from any bearing.")
        return

    # ------------------------------------------------------------------
    # Step 2: Aggregate to sub-minute resolution (30-second hop)
    # ------------------------------------------------------------------
    all_aggregated = {}
    for bearing_id, df_seg in tqdm(
        all_segment_features.items(),
        desc="Phase 2: Sub-Minute Aggregation"
    ):
        df_agg = aggregate_with_subminute_hop(df_seg)
        all_aggregated[bearing_id] = df_agg

    # ------------------------------------------------------------------
    # Step 3: Construct ground truth
    # ------------------------------------------------------------------
    all_with_targets = {}
    all_metadata = []

    for bearing_id, df_agg in tqdm(
        all_aggregated.items(),
        desc="Phase 3: Ground Truth Construction"
    ):
        df_gt, metadata = construct_ground_truth(df_agg)
        all_with_targets[bearing_id] = df_gt
        all_metadata.append(metadata)

    # Save metadata for analysis
    meta_df = pd.DataFrame(all_metadata)
    meta_df.to_csv(
        os.path.join(OUTPUT_PATH, "bearing_metadata.csv"), index=False
    )
    print(f"\nMetadata saved. {len(meta_df)} bearings processed.")

    # ------------------------------------------------------------------
    # Step 4, 5 & 6: Fold-based Global Scaling and Windowing
    # ------------------------------------------------------------------
    sample_df = next(iter(all_with_targets.values()))
    non_feature_cols = [
        "Time_Index", "Bearing_ID", "Change_Point",
        "Health_Index", "Target_RUL"
    ]
    agg_feat_cols = sorted(
        [c for c in sample_df.columns if c not in non_feature_cols]
    )

    folds = get_stratified_folds()

    for w_size in WINDOW_SIZES:
        print(f"\n--- Window Size: {w_size} "
              f"(covers {w_size * 0.5:.1f} minutes) ---")

        w_dir = os.path.join(OUTPUT_PATH, f"window_size_{w_size}")
        os.makedirs(w_dir, exist_ok=True)

        for fold_idx, cv_split in enumerate(folds, start=1):
            # 1. Gather all training data for this fold to fit the scaler
            train_dfs_raw = [
                all_with_targets[b] for b in cv_split["train"]
                if b in all_with_targets
            ]
            if not train_dfs_raw:
                continue
            
            combined_train = pd.concat(train_dfs_raw, ignore_index=True)
            
            # 2. Fit StandardScaler globally across all training segments
            scaler = StandardScaler()
            scaler.fit(combined_train[agg_feat_cols])
            
            # 3. Transform train and val with this scaler, then slide windows
            val_list = []
            for b in cv_split["val"]:
                if b in all_with_targets:
                    df_s = all_with_targets[b].copy()
                    df_s[agg_feat_cols] = scaler.transform(df_s[agg_feat_cols])
                    w_df = create_sliding_windows(df_s, w_size)
                    if not w_df.empty:
                        val_list.append(w_df)
                        
            train_list = []
            for b in cv_split["train"]:
                if b in all_with_targets:
                    df_s = all_with_targets[b].copy()
                    df_s[agg_feat_cols] = scaler.transform(df_s[agg_feat_cols])
                    w_df = create_sliding_windows(df_s, w_size)
                    if not w_df.empty:
                        train_list.append(w_df)

            # 4. Save to CSV
            if val_list:
                val_df = pd.concat(val_list, ignore_index=True)
                val_path = os.path.join(
                    w_dir, f"processed_val_fold{fold_idx}.csv"
                )
                val_df.to_csv(val_path, index=False)
                print(f"  Fold {fold_idx} val: {len(val_df)} samples -> {os.path.basename(val_path)}")

            if train_list:
                train_df = (
                    pd.concat(train_list, ignore_index=True)
                    .sample(frac=1, random_state=42)
                    .reset_index(drop=True)
                )
                train_path = os.path.join(
                    w_dir, f"processed_train_fold{fold_idx}.csv"
                )
                train_df.to_csv(train_path, index=False)
                print(f"  Fold {fold_idx} train: {len(train_df)} samples -> {os.path.basename(train_path)}")

    print("\n" + "=" * 60)
    print("PREPROCESSING COMPLETE")
    print("=" * 60)

In [ ]:
# ============================================================================
# EXECUTION
# ============================================================================
if __name__ == "__main__":
    preprocess_and_slide()